In [ ]:
import sys
print(sys.executable)


In [ ]:
import torch
import numpy as np
import joblib

RESULTS_DIR = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results'
SAVE_DIR    = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\data\processed'

# ML models — sab joblib se load
xgb_model  = joblib.load(f'{RESULTS_DIR}\\xgboost_model.pkl')
lgbm_model = joblib.load(f'{RESULTS_DIR}\\lightgbm_model.pkl')
cat_model  = joblib.load(f'{RESULTS_DIR}\\catboost_model.pkl')
scaler     = joblib.load(f'{RESULTS_DIR}\\scaler.pkl')
tau        = np.load(f'{RESULTS_DIR}\\tau.npy')[0]

# Autoencoder
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class VoiceAutoencoder(torch.nn.Module):
    def __init__(self, input_dim=61):
        super(VoiceAutoencoder, self).__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 32), torch.nn.ReLU(),
            torch.nn.Linear(32, 16),        torch.nn.ReLU(),
            torch.nn.Linear(16, 8),         torch.nn.ReLU()
        )
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(8, 16),         torch.nn.ReLU(),
            torch.nn.Linear(16, 32),        torch.nn.ReLU(),
            torch.nn.Linear(32, input_dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae_model = VoiceAutoencoder(input_dim=61).to(device)
ae_model.load_state_dict(torch.load(f'{RESULTS_DIR}\\autoencoder.pth'))
ae_model.eval()

print("All models loaded ✅")
print(f"Threshold tau: {tau:.4f}")
print(f"Device: {device}")

In [ ]:
import librosa
import numpy as np
import pandas as pd

def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, sr=16000)
        mfcc     = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).mean(axis=1)
        rolloff  = librosa.feature.spectral_rolloff(y=y, sr=sr).mean()
        zcr      = librosa.feature.zero_crossing_rate(y).mean()
        chroma   = librosa.feature.chroma_stft(y=y, sr=sr).mean(axis=1)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr).mean(axis=1)
        return np.concatenate([mfcc, [rolloff], [zcr], chroma, contrast])
    except:
        return None

def two_gate_predict(file_path):
    # Feature extraction
    feat = extract_features(file_path)
    if feat is None:
        return "ERROR"
    
    feat_sc = scaler.transform([feat])

    # Gate 1 — ML Ensemble voting
    p_xgb  = xgb_model.predict(feat_sc)[0]
    p_lgbm = lgbm_model.predict(feat_sc)[0]
    p_cat  = cat_model.predict(feat_sc)[0]
    votes  = p_xgb + p_lgbm + p_cat

    if votes >= 2:
        return "🚨 KNOWN SPOOF"

    # Gate 2 — Autoencoder
    feat_tensor   = torch.FloatTensor(feat_sc).to(device)
    reconstructed = ae_model(feat_tensor)
    error = ((feat_tensor - reconstructed) ** 2).mean().item()

    if error > tau:
        return f"⚠️  UNKNOWN SUSPICIOUS (error={error:.3f})"
    
    return "✅ BONAFIDE"

# Test on sample files
DATA_ROOT   = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\data\raw\LA\LA'
PROTOCOL_DIR = f'{DATA_ROOT}\\ASVspoof2019_LA_cm_protocols'
EVAL_AUDIO  = f'{DATA_ROOT}\\ASVspoof2019_LA_eval\\flac'

eval_df = pd.read_csv(f'{PROTOCOL_DIR}\\ASVspoof2019.LA.cm.eval.trl.txt',
    sep=' ', header=None,
    names=['speaker_id', 'file_id', 'env', 'attack_id', 'label'])

import pandas as pd

# 5 real + 5 fake test
test_bon  = eval_df[eval_df['label'] == 'bonafide'].head(5)
test_sp   = eval_df[eval_df['label'] == 'spoof'].head(5)
test_df   = pd.concat([test_bon, test_sp])

print("=== Two-Gate Pipeline Test ===\n")
for _, row in test_df.iterrows():
    file_path = f'{EVAL_AUDIO}\\{row["file_id"]}.flac'
    result    = two_gate_predict(file_path)
    actual    = row['label'].upper()
    print(f"Actual: {actual:10} → Predicted: {result}")

In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report

results = []
actuals = []

# 500 sample test — speed check
test_sample = eval_df.sample(500, random_state=42)

for _, row in tqdm(test_sample.iterrows(), total=500):
    file_path = f'{EVAL_AUDIO}\\{row["file_id"]}.flac'
    result    = two_gate_predict(file_path)
    actual    = 1 if row['label'] == 'spoof' else 0
    
    # Convert result to binary
    if '✅' in result:
        pred = 0  # bonafide
    else:
        pred = 1  # spoof or suspicious

    results.append(pred)
    actuals.append(actual)

print("\n=== Two-Gate — Eval Set (500 samples) ===")
print(classification_report(actuals, results,
      target_names=['bonafide', 'spoof'], zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Results summary
summary = {
    'Evaluation':    ['Dev Set\n(Known A01-A06)', 
                      'Eval Set\n(Unknown A07-A19)',
                      'Two-Gate\n(Eval Set)'],
    'Spoof Recall':  [0.99, 0.70, 0.72],
    'Bonafide Recall':[0.95, 0.96, 0.94],
    'Accuracy':      [0.99, 0.72, 0.74]
}

fig, ax = plt.subplots(figsize=(12, 5))

x     = np.arange(len(summary['Evaluation']))
width = 0.25

bars1 = ax.bar(x - width, summary['Accuracy'],       width, label='Accuracy',        color='steelblue', alpha=0.85)
bars2 = ax.bar(x,         summary['Spoof Recall'],   width, label='Spoof Recall',    color='red',       alpha=0.85)
bars3 = ax.bar(x + width, summary['Bonafide Recall'],width, label='Bonafide Recall', color='green',     alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, 
                bar.get_height() + 0.01,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(summary['Evaluation'])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Complete System — Performance Across All Evaluation Scenarios',
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\figures\final_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Final comparison saved ✅")